# Fan-Sim Colab - A100 GPU Training

Use this notebook on Colab Pro with an A100 runtime after graph data has been synced to Google Drive. It restores the graph data and trains the PhysicsNeMo backend from configs/fan_sim_colab.yaml.

In [ ]:
# Mount Google Drive for persistent input, checkpoints, and output sync.
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import os
import shutil
import subprocess

REPO_URL = 'https://github.com/tcthuong/fan-sim.git'
PROJECT_ROOT = Path('/content/fan-sim')
DRIVE_ROOT = Path('/content/drive/MyDrive/colab-data/fan-sim')
CHECKPOINT_DIR = DRIVE_ROOT / 'checkpoints'
SYNC_ROOT = DRIVE_ROOT / 'outputs'

if not Path('/content/drive/MyDrive').exists():
    raise RuntimeError('Mount Google Drive before configuring checkpoints and sync.')

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
SYNC_ROOT.mkdir(parents=True, exist_ok=True)


def _now_iso():
    return datetime.now(timezone.utc).isoformat()


def _has_data(path):
    path = Path(path)
    if not path.exists():
        return False
    if path.is_file():
        return path.stat().st_size > 0
    return any(path.iterdir())


def _project_path(path):
    path = Path(path)
    if path.is_absolute():
        return path
    return PROJECT_ROOT / path


def _sync_destination(src):
    src = Path(src)
    try:
        rel = src.resolve().relative_to(PROJECT_ROOT.resolve())
    except ValueError:
        rel = Path(src.name)
    return SYNC_ROOT / rel


def _copy_tree_or_file(src, dst, delete=True):
    src = Path(src)
    dst = Path(dst)
    dst.parent.mkdir(parents=True, exist_ok=True)
    if src.is_file():
        shutil.copy2(src, dst)
    elif shutil.which('rsync'):
        dst.mkdir(parents=True, exist_ok=True)
        args = ['rsync', '-a']
        if delete:
            args.append('--delete')
        args.extend([f'{src}/', f'{dst}/'])
        subprocess.run(args, check=True)
    else:
        if delete and dst.exists():
            shutil.rmtree(dst)
        shutil.copytree(src, dst, dirs_exist_ok=not delete)


def sync_to_drive(*paths):
    synced = []
    for raw_path in paths:
        src = _project_path(raw_path)
        if not _has_data(src):
            print(f'[sync] skip empty/missing: {src}')
            continue

        dst = _sync_destination(src)
        _copy_tree_or_file(src, dst, delete=True)
        synced.append(str(dst))
        print(f'[sync] {src} -> {dst}')
    return synced


def sync_from_drive(*paths):
    restored = []
    for raw_path in paths:
        dst = _project_path(raw_path)
        src = _sync_destination(dst)
        if not _has_data(src):
            raise FileNotFoundError(f'No synced data in Google Drive: {src}')

        _copy_tree_or_file(src, dst, delete=True)
        restored.append(str(dst))
        print(f'[restore] {src} -> {dst}')
    return restored


def mark_checkpoint(step, **metadata):
    checkpoint = CHECKPOINT_DIR / f'{step}.json'
    payload = {
        'step': step,
        'completed_at': _now_iso(),
        'project_root': str(PROJECT_ROOT),
        'sync_root': str(SYNC_ROOT),
        'metadata': metadata,
    }
    tmp = checkpoint.with_suffix('.json.tmp')
    tmp.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding='utf-8')
    tmp.replace(checkpoint)
    print(f'[checkpoint] {checkpoint}')
    return checkpoint


def checkpoint_and_sync(step, *paths, **metadata):
    checkpoint = mark_checkpoint(step, **metadata)
    synced = sync_to_drive(*paths)
    return {'checkpoint': str(checkpoint), 'synced': synced}


def run(command, cwd=PROJECT_ROOT):
    cwd = Path(cwd)
    if not cwd.exists():
        print(f'[run] cwd missing, using /content instead: {cwd}')
        cwd = Path('/content')
    print(f'$ {command}')
    subprocess.run(command, shell=True, check=True, cwd=str(cwd), executable='/bin/bash')


def run_step(step, commands, sync_paths=(), **metadata):
    for command in commands:
        run(command)
    return checkpoint_and_sync(step, *sync_paths, **metadata)

print(f'Checkpoint dir: {CHECKPOINT_DIR}')
print(f'Sync dir: {SYNC_ROOT}')

## A100 GPU training

Switch to a Colab Pro A100 runtime if needed. This section restores graph data from Google Drive into the local runtime and trains the PhysicsNeMo backend from configs/fan_sim_colab.yaml without regenerating cases, VTU, or graphs.

### Pull GitHub repo

Run this step in every fresh Colab runtime. It clones the repo if missing, otherwise pulls the latest code with `git pull --ff-only`.

In [ ]:
# Pull GitHub repo: clone on a new runtime, pull when the repo already exists.
if (PROJECT_ROOT / '.git').is_dir():
    run(f'git -C "{PROJECT_ROOT}" pull --ff-only', cwd=Path('/content'))
else:
    if PROJECT_ROOT.exists():
        shutil.rmtree(PROJECT_ROOT)
    run(f'git clone "{REPO_URL}" "{PROJECT_ROOT}"', cwd=Path('/content'))

os.chdir(PROJECT_ROOT)
print(Path.cwd())
run('python --version')
checkpoint_and_sync('10_gpu_repo_ready')

In [ ]:
# Verify this runtime has GPU after the repo is ready.
run('nvidia-smi')

In [ ]:
# GPU processing uses synced data from Drive. Do not regenerate cases or graphs here.
sync_from_drive('artifacts/graphs')
run('find artifacts/graphs -name "*.graph.pt" -print')
mark_checkpoint('11_gpu_data_restored', restored=['artifacts/graphs'])

In [ ]:
# Heavy ML install for PhysicsNeMo MeshGraphNet.
run_step(
    '12_physicsnemo_install',
    [
        'python -m pip install --upgrade pip',
        'python -m pip install -e ".[ml]"',
        'python -c "from physicsnemo.models.meshgraphnet.meshgraphnet import MeshGraphNet; print(\"MeshGraphNet ok\")"',
    ],
)

In [ ]:
# Verify the Colab config is set for A100/PhysicsNeMo training.
import yaml
cfg = yaml.safe_load((PROJECT_ROOT / 'configs/fan_sim_colab.yaml').read_text())
assert cfg['model']['backend'] == 'physicsnemo', cfg['model']
print('Training config: configs/fan_sim_colab.yaml')
print(cfg['model'])
mark_checkpoint('13_gpu_config_ready', config='configs/fan_sim_colab.yaml', model=cfg['model'])

In [ ]:
run_step(
    '14_train_gpu_model',
    [
        'fan-sim train --config configs/fan_sim_colab.yaml --epochs 1',
        'find artifacts/models/fan_mgn_colab_a100 -maxdepth 2 -type f -print',
    ],
    sync_paths=['artifacts/models/fan_mgn_colab_a100'],
)